# Fine-tune LLM with Ray on SageMaker HyperPod (Studio Workflow)

**FSDP + LoRA on a Ray cluster managed through SageMaker Studio**

This notebook uses the new **Ray on HyperPod** integration in SageMaker Studio. Studio handles
cluster creation, Ray Dashboard access, Amazon Managed Grafana observability, workspace
connectivity, and hung job detection — no kubectl, Helm, or Kubernetes YAML required.

### What You Do vs. What Studio Does

| You | Studio |
|-----|--------|
| Write training script (`train_ray.py`) | Creates Ray cluster via KubeRay operator |
| Configure hyperparameters (`args.yaml`) | Installs CRDs (RayCluster, RayJob, RayService) |
| Prepare dataset | Creates namespace, service account, FSx storage |
| Submit and monitor job | Provides Ray Dashboard with IAM-authenticated URLs |
| | Provisions 4 Grafana dashboards (Core, Data, Train, Serve) |
| | Connects JupyterLab/Code Editor as zero-compute worker |
| | Detects hung jobs via per-node Job Monitoring Agent |
| | Supports tiered checkpointing (local disk → S3) |

## Prerequisites

Before running this notebook, ensure four components are installed on your HyperPod EKS cluster:

| Component | Purpose |
|-----------|--------|
| **SageMaker Spaces EKS add-on** | Enables JupyterLab and Code Editor workspaces that attach to Ray clusters |
| **HyperPod Observability EKS add-on** | Collects metrics from Ray workloads, provisions Grafana dashboards |
| **KubeRay operator** | Manages RayCluster, RayJob, and RayService as native Kubernetes resources |
| **HyperPod Ray Endpoint Operator** (Helm chart) | Generates authenticated public endpoints for dashboard and remote job submission |

You also need:
- A **SageMaker Studio domain** with access to the HyperPod cluster
- **FSx for Lustre** file system attached to the cluster

See the [Ray on HyperPod getting started guide](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-hyperpod-eks-ray.html) for full setup details.

## Step 1: Create a Ray Cluster from SageMaker Studio

1. Open **SageMaker Studio**
2. Navigate to **HyperPod** in the left sidebar
3. Select your HyperPod cluster
4. Go to the **Tasks** tab
5. From the task type list, choose **RayCluster**
6. Click **Create Ray Cluster** and configure:
   - **Cluster name**: `ray-medical-finetune`
   - **Head instance type**: select an instance (no GPU needed for head)
   - **Worker instance type**: any GPU instance (e.g., `g5.xlarge` — 1 GPU is sufficient for Qwen3-0.6B with LoRA)
   - **Worker count**: 1 (scale up for larger models — see README)
   - **Container image**: use the default **SageMaker Distribution image** (Ray pre-installed, AWS-maintained with vulnerability patching)
7. Under advanced settings, **enable remote endpoints** for dashboard access and remote job submission
8. Click **Create**

Studio automatically:
- Registers CRDs (RayCluster, RayJob, RayService) via the KubeRay operator
- Creates the namespace, service account, and FSx PV/PVC
- Deploys the RayCluster with your configuration
- Integrates with HyperPod task governance for compute quotas and scheduling priorities

Wait for the cluster status to show **Running** on the Tasks tab before continuing.

> **Tip**: For advanced customization, Studio provides an inline YAML editor that exposes the full
> Kubernetes manifest. Use this if you need custom tolerations, node selectors, or resource overrides.

> **Scaling**: The default config uses 1 worker for the 0.6B model. For larger models (7B+),
> increase the worker count and update `num_workers` in `args.yaml` to match.

## Step 2: Access the Ray Dashboard

Once the cluster is **Running**:

1. On the **Tasks** tab, find your Ray cluster
2. Click **Actions** → **Open Ray Dashboard**
3. Studio generates a **short-lived, IAM-authenticated URL** scoped to the cluster creator
4. The dashboard opens in a new browser tab showing cluster health, running jobs, and node status

No `kubectl port-forward` needed. The URL is secured via IAM and scoped to your identity.

## Step 3: Connect a Workspace to the Ray Cluster

1. In Studio, create a **HyperPod JupyterLab** or **Code Editor** space
2. During space creation, a **Ray cluster integration option** appears in the configuration panel
3. Select your Ray cluster (`ray-medical-finetune`)
4. The space restarts to attach to the cluster
5. Open the workspace in your browser

The workspace joins the cluster as a **zero-compute worker node**, giving you full native
Ray driver access. You work as if on the head node — `ray.init(address="auto")` connects
directly to the cluster.

**Open this notebook in the connected workspace to continue.**

## Step 4: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import ray

# Connect to the attached Ray cluster. The workspace joined as a zero-compute
# worker node during Step 3, so address="auto" finds the cluster automatically.
ray.init(address="auto")

print(f"Ray version: {ray.__version__}")
print(f"Available resources: {ray.cluster_resources()}")
print(f"Nodes: {len(ray.nodes())}")
ray.shutdown()

### Install dependencies on Ray workers

Ray's `runtime_env` parameter injects Python dependencies at runtime without rebuilding
container images. We'll use this when submitting the job in Step 7. No need to SSH into
workers or run `kubectl exec`.

## Step 5: Prepare the Dataset

FSx for Lustre is mounted directly at `/mnt/fsx` in both the workspace and all Ray workers.
Write data there — no `kubectl cp` needed.

In [ ]:
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_ID = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES = 1000

dataset = load_dataset(DATASET_ID, "en", split=f"train[:{NUM_SAMPLES}]")
print(f"Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nExample:\n{dataset[0]}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = """You are a medical reasoning assistant. Respond to user queries
with clear, step-by-step medical reasoning."""


def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["Question"]},
        {
            "role": "assistant",
            "content": f"<think>\n{example['Complex_CoT']}\n</think>\n\n{example['Response']}",
        },
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example


dataset = dataset.map(format_chat)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
import os

TRAIN_PATH = "/mnt/fsx/datasets/medical-o1/train"
VAL_PATH = "/mnt/fsx/datasets/medical-o1/val"

os.makedirs(TRAIN_PATH, exist_ok=True)
os.makedirs(VAL_PATH, exist_ok=True)

train_dataset.to_json(os.path.join(TRAIN_PATH, "train.json"))
val_dataset.to_json(os.path.join(VAL_PATH, "val.json"))

print(f"Train data written to: {TRAIN_PATH}")
print(f"Val data written to:   {VAL_PATH}")

## Step 6: Copy Training Script and Config to FSx

Ray workers access files from the shared FSx mount. Copy the training files there directly
from this workspace.

In [ ]:
import shutil

SCRIPTS_PATH = "/mnt/fsx/scripts"
os.makedirs(SCRIPTS_PATH, exist_ok=True)

shutil.copy("train_ray.py", os.path.join(SCRIPTS_PATH, "train_ray.py"))
shutil.copy("args.yaml", os.path.join(SCRIPTS_PATH, "args.yaml"))
shutil.copy("requirements.txt", os.path.join(SCRIPTS_PATH, "requirements.txt"))

print(f"Files copied to {SCRIPTS_PATH}:")
print(os.listdir(SCRIPTS_PATH))

## Step 7: Submit the Training Job

Three options for job submission:

- **Option A**: Ray Jobs API from this notebook (programmatic)
- **Option B**: Remote CLI via `toolkit-for-ray-on-sagemaker-ai` (from laptop or CI/CD)
- **Option C**: `ray job submit` from the workspace terminal

### Resilience

The training script uses `FailureConfig(max_failures=3)`. If a node fails:
1. HyperPod automatically replaces the faulty node
2. Ray reschedules worker pods onto the new node
3. Training resumes from the last checkpoint

No changes to your training code are needed for this to work.

### Option A: Ray Jobs API (from this notebook)

In [ ]:
from ray.job_submission import JobSubmissionClient

# Studio auto-configures RAY_ADDRESS when the workspace is attached to the cluster.
RAY_ADDRESS = os.environ.get("RAY_ADDRESS", "http://localhost:8265")

client = JobSubmissionClient(RAY_ADDRESS)
print(f"Connected to Ray at: {RAY_ADDRESS}")
print(f"Existing jobs: {len(client.list_jobs())}")

In [ ]:
job_id = client.submit_job(
    entrypoint="python /mnt/fsx/scripts/train_ray.py --config /mnt/fsx/scripts/args.yaml",
    runtime_env={
        "working_dir": "/mnt/fsx/scripts",
        "pip": "/mnt/fsx/scripts/requirements.txt",
        "env_vars": {
            "HF_HOME": "/mnt/fsx/huggingface",
            "NCCL_DEBUG": "INFO",
            "NCCL_SOCKET_IFNAME": "eth0",
        },
    },
    metadata={
        "name": "qwen3-medical-finetune",
        "cluster": "hyperpod-studio",
    },
)

print(f"Job submitted: {job_id}")
print(f"\nView in Ray Dashboard: Actions > Open Ray Dashboard in Studio")
print(f"View in Grafana:       Actions > Open Grafana in Studio")

### Option B: Remote CLI (from laptop or CI/CD)

The `toolkit-for-ray-on-sagemaker-ai` package enables remote job submission from anywhere
with IAM credentials. It handles endpoint resolution and EKS API credential generation
through IAM authentication.

```bash
# One-time setup
pip install toolkit-for-ray-on-sagemaker-ai
aws eks update-kubeconfig --name <eks-cluster-name> --region <region>

# Submit job using the sagemaker_ray:// address scheme
ray job submit \
    --address sagemaker_ray://<ray-cluster-name>/<namespace> \
    --working-dir . \
    -- python train_ray.py --config args.yaml

# List jobs
ray job list --address sagemaker_ray://<ray-cluster-name>/<namespace>
```

Replace `<ray-cluster-name>` and `<namespace>` with your cluster's values from the Studio Tasks tab.

### Option C: Workspace terminal

Open a terminal in this JupyterLab workspace and run:

```bash
ray job submit \
    --working-dir /mnt/fsx/scripts \
    --runtime-env-json '{"pip": "/mnt/fsx/scripts/requirements.txt", "env_vars": {"HF_HOME": "/mnt/fsx/huggingface", "NCCL_DEBUG": "INFO"}}' \
    -- python /mnt/fsx/scripts/train_ray.py --config /mnt/fsx/scripts/args.yaml
```

## Step 8: Monitor the Job

### From Studio (recommended)

Three monitoring surfaces, all accessible from the **Tasks tab Actions menu**:

1. **Ray Dashboard** — Click **Open Ray Dashboard**. Shows job status, worker logs,
   resource utilization, and task timeline. Uses IAM-authenticated short-lived URLs.

2. **Amazon Managed Grafana** — Click **Open Grafana**. The HyperPod Observability add-on
   auto-discovers Ray pods, scrapes their metrics endpoints, and provisions **four pre-built
   dashboards** under a Ray folder:
   - **Ray Core** — cluster-level resource utilization
   - **Ray Data** — data loading and preprocessing metrics
   - **Ray Train** — training throughput, loss curves, GPU utilization
   - **Ray Serve** — inference endpoint metrics

   Dashboards support filtering by specific Ray cluster and appear alongside existing
   HyperPod infrastructure dashboards (GPU, EFA, task governance).

3. **Hung job detection** — The per-node **Job Monitoring Agent** monitors node-level and
   job-level signals to detect when training has stalled (e.g., a pod fails and other pods
   block at the next collective operation). No code changes required for Ray Train workloads.
   Notifications are sent to:
   - The cluster's **Amazon CloudWatch log group**
   - The **Ray Train Grafana dashboard**

### From this notebook (programmatic)

In [ ]:
import time

while True:
    status = client.get_job_status(job_id)
    print(f"Status: {status}")
    if status.is_terminal():
        break
    time.sleep(30)

logs = client.get_job_logs(job_id)
print(f"\n{'=' * 60}")
print(f"Job Logs (last 2000 chars):")
print(f"{'=' * 60}")
print(logs[-2000:])

## Step 9: Configure Hung Job Detection (Optional)

The built-in hung job detection works out of the box for Ray Train workloads. For custom
detection rules (specific log patterns or timeout thresholds), use the
`toolkit-for-ray-on-sagemaker-ai` library.

When the configured action is `cancel`, HyperPod terminates the hung worker process, and
Ray Train's `FailureConfig` restarts workers from the last checkpoint automatically.

In [ ]:
# Example: custom hung job detection rules (optional)
# The API below is illustrative — refer to the toolkit-for-ray-on-sagemaker-ai
# documentation for the exact module and method signatures.

# from toolkit_for_ray_on_sagemaker_ai import hung_job_detection
#
# hung_job_detection.configure(
#     cluster_name="ray-medical-finetune",
#     rules=[
#         {
#             "name": "no-gpu-activity",
#             "log_pattern": "GPU utilization: 0%",
#             "timeout_seconds": 600,
#             "action": "cancel",
#         },
#     ],
# )

## Step 10: Verify Output

Model artifacts are on FSx — directly accessible from this workspace.

In [ ]:
OUTPUT_DIR = "/mnt/fsx/output/qwen3-medical"

print("Output directory:")
if os.path.exists(OUTPUT_DIR):
    for item in os.listdir(OUTPUT_DIR):
        item_path = os.path.join(OUTPUT_DIR, item)
        if os.path.isdir(item_path):
            size = sum(f.stat().st_size for f in os.scandir(item_path) if f.is_file())
            print(f"  {item}/ ({size / 1e9:.2f} GB)")
        else:
            print(f"  {item} ({os.path.getsize(item_path) / 1e6:.1f} MB)")
else:
    print("  Output directory not found — job may still be running.")

In [ ]:
from transformers import pipeline

MERGED_PATH = "/mnt/fsx/output/qwen3-medical/merged_model"

if os.path.exists(MERGED_PATH):
    print("Loading merged model for validation...")
    pipe = pipeline(
        "text-generation",
        model=MERGED_PATH,
        torch_dtype="auto",
        device_map="auto",
    )

    test_prompt = "What are the common symptoms and diagnostic criteria for Type 2 Diabetes?"
    result = pipe(
        [{"role": "user", "content": test_prompt}],
        max_new_tokens=512,
    )
    print(f"\nPrompt: {test_prompt}")
    print(f"\nResponse:\n{result[0]['generated_text'][-1]['content']}")
else:
    print(f"Merged model not found at {MERGED_PATH}")
    print("Check if merge_weights is enabled in args.yaml and job completed.")

## Step 11: Upload Model to S3 (Optional)

Copy the final model to S3 for deployment via SageMaker Endpoints or other services.

In [ ]:
import boto3

S3_BUCKET = "your-model-bucket"  # <-- Replace with your bucket
S3_PREFIX = "models/qwen3-medical-finetuned"

if os.path.exists(MERGED_PATH):
    s3 = boto3.client("s3")
    for root, dirs, files in os.walk(MERGED_PATH):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, MERGED_PATH)
            s3_key = f"{S3_PREFIX}/{relative_path}"
            print(f"Uploading: {relative_path}")
            s3.upload_file(local_path, S3_BUCKET, s3_key)

    print(f"\nModel uploaded to: s3://{S3_BUCKET}/{S3_PREFIX}/")
else:
    print("Merged model not found. Skipping S3 upload.")

## Step 12: Scale and Iterate

One of the key advantages of this workflow: you can scale without recreating the cluster.

- **Scale workers** from Studio: edit the cluster and change the worker count
- **Scale from code**: change `num_workers` in `args.yaml` — prototype with 1 worker, then
  scale to 8+ by changing one line in `ScalingConfig`
- **The notebook stays interactive** while training runs distributed across the cluster —
  monitor, adjust hyperparameters, or inspect intermediate results

## Step 13: Clean Up

Manage the Ray cluster from Studio:

1. On the **Tasks** tab, select your Ray cluster
2. Click **Actions** → **Delete**

HyperPod nodes remain available for other workloads after the Ray cluster is deleted.
To stop incurring compute charges entirely, delete the HyperPod cluster from the SageMaker AI console.

In [ ]:
# Optionally clean up dataset and script files from FSx

# import shutil
# shutil.rmtree("/mnt/fsx/datasets/medical-o1", ignore_errors=True)
# shutil.rmtree("/mnt/fsx/scripts", ignore_errors=True)
# shutil.rmtree("/mnt/fsx/output/qwen3-medical", ignore_errors=True)
# shutil.rmtree("/mnt/fsx/checkpoints/qwen3-medical", ignore_errors=True)
# shutil.rmtree("/mnt/fsx/ray_results", ignore_errors=True)
# print("Cleaned up FSx working files.")

print("Cleanup commands above are commented out for safety. Uncomment to execute.")

## Summary

| Step | What happened |
|------|---------------|
| 1 | Created Ray cluster from SageMaker Studio Tasks tab |
| 2 | Opened Ray Dashboard via IAM-authenticated URL |
| 3 | Connected JupyterLab workspace (zero-compute worker node) |
| 4 | Installed Python dependencies |
| 5 | Prepared dataset and wrote directly to FSx |
| 6 | Copied training script to FSx |
| 7 | Submitted Ray training job (3 submission options) |
| 8 | Monitored via Ray Dashboard, Managed Grafana, and hung job detection |
| 9 | (Optional) Configured custom hung job detection rules |
| 10 | Verified model output on FSx |
| 11 | Uploaded model to S3 |
| 12 | Scaled workers without recreating the cluster |
| 13 | Cleaned up Ray cluster from Studio |

### What We Didn't Have to Do

- Install or configure `kubectl` or `helm`
- Write Kubernetes YAML manifests (RayCluster, PV, PVC, Namespace, ServiceAccount)
- Install CRDs (KubeRay operator, Kubeflow Training Operator)
- Debug pod scheduling, node capacity, or webhook failures
- Set up `kubectl port-forward` for the Ray Dashboard
- Configure Prometheus and Grafana for observability
- Rebuild Docker images for dependency changes
- Use `kubectl cp` or `kubectl exec` to move files and install packages
- Configure PodMonitors, ServiceMonitors, or scrape targets
- Set up IAM roles for SigV4 signing

### Resilience Features (automatic)

| Layer | What it does |
|-------|-------------|
| **Node recovery** | HyperPod replaces faulty nodes; Ray reschedules pods; `FailureConfig` retries the job |
| **Hung job detection** | Per-node Job Monitoring Agent detects stalls; alerts via CloudWatch and Grafana |
| **Tiered checkpointing** | `amzn-sagemaker-checkpointing` writes to local disk first, async uploads to S3 |

### Key Packages

| Package | Purpose |
|---------|--------|
| `toolkit-for-ray-on-sagemaker-ai` | Remote job submission (`sagemaker_ray://` address), JumpStart model loader, custom hung job detection rules |
| `amzn-sagemaker-checkpointing` | Tiered checkpointing integration (local disk → S3) |
| SageMaker Distribution image | Default container with Ray pre-installed (AWS-maintained) |